In [10]:
# HPV STRATIFICATION ANALYSIS
print("HPV+ vs HPV- MUTATION LANDSCAPE")
print("%" * 50)


import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests


# Load data
mutations = pd.read_csv('../data/clean_mutations.tsv', sep='\t')
clinical = pd.read_csv("../data/clean_clinical.tsv", sep='\t')

print(f"Mutations: {mutations.shape}, Clinical: {clinical.shape}")


# Filter non-hypermutators
non_hyper_mut = mutations[mutations['hypermutator'] != 1]
non_hyper_clin = clinical[clinical['hypermutator'] != 1]

print(f"\nNon-hypermutators: {len(non_hyper_mut['SAMPLE_ID'].unique())}")


# HPV groups (p16 only)
hpv_pos = non_hyper_clin[non_hyper_clin['HPV_clean'] == 'Positive']['SAMPLE_ID']
hpv_neg = non_hyper_clin[non_hyper_clin['HPV_clean'] == 'Negative']['SAMPLE_ID']


n_hpv_pos = len(hpv_pos)
n_hpv_neg = len(hpv_neg)
print(f"HPV+ samples: {n_hpv_pos}, HPV- Clinical: {n_hpv_neg}")


# Verify these samples have mutations
hpv_pos_mut = hpv_pos[hpv_pos.isin(non_hyper_mut['SAMPLE_ID'])]
hpv_neg_mut = hpv_neg[hpv_neg.isin(non_hyper_mut['SAMPLE_ID'])]
print(f"HPV+ WITH mutations: {len(hpv_pos_mut)}")
print(f"HPV- WITH mutations: {len(hpv_neg_mut)}")

HPV+ vs HPV- MUTATION LANDSCAPE
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
Mutations: (84690, 12), Clinical: (528, 37)

Non-hypermutators: 504
HPV+ samples: 41, HPV- Clinical: 74
HPV+ WITH mutations: 35
HPV- WITH mutations: 73


In [12]:
# Top 15 genes
gene_counts = non_hyper_mut.groupby('Hugo_Symbol')['SAMPLE_ID'].nunique()
top_genes = gene_counts.sort_values(ascending=False).head(15).index.tolist()
print(f"Top 15 genes: {top_genes[:5]}")
print("Gene types:", [type(g) for g in top_genes[:3]])


# Frequencies by HPV status
hpv_freq = []
for gene in top_genes:
    # HPV+ frequency
    hpv_pos_mut_gene = non_hyper_mut[(non_hyper_mut['SAMPLE_ID'].isin(hpv_pos_mut)) & (non_hyper_mut['Hugo_Symbol'] == gene)]['SAMPLE_ID'].nunique()
    freq_pos = hpv_pos_mut_gene / len(hpv_pos_mut) * 100

    # HPV- frequency
    hpv_neg_mut_gene = non_hyper_mut[(non_hyper_mut['SAMPLE_ID'].isin(hpv_neg_mut)) & (non_hyper_mut['Hugo_Symbol'] == gene)]['SAMPLE_ID'].nunique()
    freq_neg = hpv_neg_mut_gene / len(hpv_neg_mut) * 100


    hpv_freq.append({
        'gene': gene,
        'hpv_pos_freq': freq_pos,
        'hpv_neg_freq': freq_neg,
        'hpv_pos_n': hpv_pos_mut_gene,
        'hpv_neg_n': hpv_neg_mut_gene
    })

hpv_df = pd.DataFrame(hpv_freq)
print(hpv_df.round(1))

Top 15 genes: ['TP53', 'TTN', 'FAT1', 'CDKN2A', 'FRG1B']
Gene types: [<class 'str'>, <class 'str'>, <class 'str'>]
      gene  hpv_pos_freq  hpv_neg_freq  hpv_pos_n  hpv_neg_n
0     TP53          14.3          84.9          5         62
1      TTN          51.4          39.7         18         29
2     FAT1           2.9          26.0          1         19
3   CDKN2A           5.7          27.4          2         20
4    FRG1B          17.1          15.1          6         11
5    CSMD3          14.3          19.2          5         14
6    MUC16          11.4          16.4          4         12
7   PIK3CA          22.9          12.3          8          9
8   NOTCH1          11.4          19.2          4         14
9    SYNE1           5.7          20.5          2         15
10   LRP1B          14.3          12.3          5          9
11   KMT2D          17.1          16.4          6         12
12    PCLO          14.3          16.4          5         12
13     FLG          14.3       

In [19]:
# Fisher's exact tests
print("\nFISHER TESTS:")
tests = []
for _, row in hpv_df.iterrows():
    gene = row['gene']
    a = row['hpv_pos_n']
    b = len(hpv_pos_mut) - a
    c = row['hpv_neg_n']
    d = len(hpv_neg_mut) - c

    odds_ratio, p_value = fisher_exact([[a, b], [c, d]])

    tests.append({
        'gene': gene, 'hpv_pos_freq': row['hpv_pos_freq'], 'hpv_neg_freq': row['hpv_neg_freq'], 'odds_ratio': odds_ratio, 'p_value': p_value
    })

tests_df = pd.DataFrame(tests)


# BH correction
rejected, q_values, _, _ = multipletests(tests_df['p_value'], method='fdr_bh', alpha=0.05)
tests_df['q_value'] = q_values
tests_df['significant'] = rejected


print(tests_df[['gene', 'hpv_pos_freq', 'hpv_neg_freq', 'odds_ratio', 'q_value']].round(4))
print(f"\nSignificant: {tests_df['significant'].sum()}")


FISHER TESTS:
      gene  hpv_pos_freq  hpv_neg_freq  odds_ratio  q_value
0     TP53       14.2857       84.9315      0.0296   0.0000
1      TTN       51.4286       39.7260      1.6065   0.7545
2     FAT1        2.8571       26.0274      0.0836   0.0226
3   CDKN2A        5.7143       27.3973      0.1606   0.0490
4    FRG1B       17.1429       15.0685      1.1661   0.9792
5    CSMD3       14.2857       19.1781      0.7024   0.9792
6    MUC16       11.4286       16.4384      0.6559   0.9792
7   PIK3CA       22.8571       12.3288      2.1070   0.5103
8   NOTCH1       11.4286       19.1781      0.5438   0.8845
9    SYNE1        5.7143       20.5479      0.2343   0.1977
10   LRP1B       14.2857       12.3288      1.1852   0.9792
11   KMT2D       17.1429       16.4384      1.0517   1.0000
12    PCLO       14.2857       16.4384      0.8472   1.0000
13     FLG       14.2857       13.6986      1.0500   1.0000
14   DNAH5       11.4286        9.5890      1.2166   0.9792

Significant: 3


In [20]:
# Where did the 6 HPV+ samples go?
hpv_pos_all = non_hyper_clin[non_hyper_clin['HPV_clean'] == 'Positive']['SAMPLE_ID']
hpv_pos_missing = hpv_pos_all[~hpv_pos_all.isin(non_hyper_mut['SAMPLE_ID'])]
print(f"HPV+ without mutation data: {len(hpv_pos_missing)}")
print(hpv_pos_missing.tolist())

HPV+ without mutation data: 6
['TCGA-BB-7861-01', 'TCGA-BB-7866-01', 'TCGA-BB-7872-01', 'TCGA-DQ-7590-01', 'TCGA-DQ-7594-01', 'TCGA-DQ-7595-01']
